<a href="https://colab.research.google.com/github/TAUforPython/LLM_AI_agents/blob/main/lesson-3_MCP_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import json
import inspect
from typing import Callable, Any, List, Tuple, Annotated, TypedDict, Literal
from openai import OpenAI

# 0. Environment setup

In [8]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-5-nano"  # Updated to a standard reliable model
client = OpenAI()

def call_llm(prompt: str) -> str:
    response = client.responses.create(model=MODEL, input=prompt)
    return response.output_text

# 1. How to build tools

In [13]:
"""# 1. How to build tools
1.1. How tool-calling works
"""
def build_initial_prompt(tools_descr: str, user_text: str) -> str:
    system_prompt = f"""
You are a medical assistant with tool-calling capabilities.
You have access to the following tools:
{tools_descr}

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "tool_call", "name": "<tool_name>", "arguments": {{...}}}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "final", "answer": "..."}}
"""
    return f"{system_prompt}\n\nUSER: {user_text}\n"

def process_tool_call(json_model_output, tools_by_name):
    tool_name = json_model_output["name"]
    tool_args = json_model_output["arguments"]
    print(f'Executing tool {tool_name} with args: {tool_args}')
    if tool_name not in tools_by_name:
        raise ValueError("Unknown tool")

    tool_result = tools_by_name[tool_name](**tool_args)
    print(f'Tool result: {tool_result}')
    return tool_result

def run_agent_inner(prompt, tools_by_name):
    model_output = call_llm(prompt)
    json_model_output = json.loads(model_output)
    print(f"model_output: {model_output}")

    if json_model_output["type"] == "tool_call":
        tool_result = process_tool_call(json_model_output, tools_by_name)
        prompt_with_tool_result = (
            f"{prompt}\n"
            f"ASSISTANT: {model_output}\n"
            f"TOOL_RESULT: {tool_result}\n"
        )
        return run_agent_inner(prompt_with_tool_result, tools_by_name)

    print("\n\n==================Last prompt==================")
    print(prompt)
    print(f"\n\n====Final result: {json_model_output['answer']}=== ")
    return json_model_output["answer"]

"""## 1.2. Building hand-crafted tools (Medical Example)"""
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI)"""
    if height_m <= 0:
        return 0.0
    return round(weight_kg / (height_m ** 2), 2)

RAW_TOOLS_BY_NAME = {'calculate_bmi': calculate_bmi}
RAW_TOOLS_DESC = """
Tool name: calculate_bmi
Description: Calculate Body Mass Index (BMI) given weight in kilograms and height in meters.
Arguments:
- weight_kg: float
- height_m: float
Returns: float
"""

def run_agent_with_raw_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(RAW_TOOLS_DESC, user_text)
    return run_agent_inner(prompt, RAW_TOOLS_BY_NAME)

# 🩺 Medical examples instead of math
run_agent_with_raw_tools("What is the BMI for a patient weighing 70 kg and 1.75 m tall?")
run_agent_with_raw_tools("Calculate BMI for 85 kg weight and 1.80 m height.")



User query: What is the BMI for a patient weighing 70 kg and 1.75 m tall?


model_output: {"type": "tool_call", "name": "calculate_bmi", "arguments": {"weight_kg": 70, "height_m": 1.75}}
Executing tool calculate_bmi with args: {'weight_kg': 70, 'height_m': 1.75}
Tool result: 22.86
model_output: {"type": "final", "answer": "The patient’s BMI is approximately 22.86 kg/m^2."}


==================Last prompt==================

You are a medical assistant with tool-calling capabilities.
You have access to the following tools:

Tool name: calculate_bmi
Description: Calculate Body Mass Index (BMI) given weight in kilograms and height in meters.
Arguments:
- weight_kg: float
- height_m: float
Returns: float


When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{"type": "final", "answer": "...

'Your BMI is 26.23 kg/m^2, which is in the Overweight category (BMI 25.0–29.9).'

# 1.3. Wrapping tools

In [10]:
"""1.3. Wrapping tools
1.3.1. Using class for unification (Fixed syntax)"""
class Tool:
    def __init__(self, name: str, description: str, func: Callable[..., Any],
                 arguments: List[Tuple[str, str]], outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        args_str = ", ".join([f"{n}: {t}" for n, t in self.arguments])
        return (f"Tool Name: {self.name}, "
                f"Description: {self.description}, "
                f"Arguments: {args_str}, "
                f"Outputs: {self.outputs}")

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

"""1.3.2. "Decorating" tools (Fixed inspect._empty)"""
def wrap_tool(func: Callable[..., Any]) -> Tool:
    sig = inspect.signature(func)
    arguments = []
    for p in sig.parameters.values():
        ann = p.annotation
        if ann is inspect.Parameter.empty:
            ann_name = "Any"
        else:
            ann_name = getattr(ann, "__name__", str(ann))
        arguments.append((p.name, ann_name))

    ret = sig.return_annotation
    if ret is inspect.Parameter.empty:
        outputs = "Any"
    else:
        outputs = getattr(ret, "__name__", str(ret))

    description = (func.__doc__ or "No description provided").strip()
    name = func.__name__
    return Tool(name, description, func, arguments, outputs)

@wrap_tool
def bmi_tool(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI) given weight in kg and height in meters."""
    return calculate_bmi(weight_kg, height_m)

bmi_tool.to_string()

"""1.3.3. All together"""
NICE_TOOLS_BY_NAME = {'bmi_tool': bmi_tool}
NICE_TOOLS_DESC = "\n".join([t.to_string() for t in NICE_TOOLS_BY_NAME.values()])

def run_agent_with_nice_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(NICE_TOOLS_DESC, user_text)
    return run_agent_inner(prompt, NICE_TOOLS_BY_NAME)

run_agent_with_nice_tools("What is the BMI for 65 kg and 1.70 m?")



User query: What is the BMI for 65 kg and 1.70 m?


model_output: {"type": "tool_call", "name": "bmi_tool", "arguments": {"weight_kg": 65, "height_m": 1.7}}
Executing tool bmi_tool with args: {'weight_kg': 65, 'height_m': 1.7}
Tool result: 22.49
model_output: {"type": "final", "answer": "Your BMI is 22.49 kg/m^2."}


==================Last prompt==================

You are a medical assistant with tool-calling capabilities.
You have access to the following tools:
Tool Name: bmi_tool, Description: Calculate Body Mass Index (BMI) given weight in kg and height in meters., Arguments: weight_kg: float, height_m: float, Outputs: float

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{"type": "final", "answer": "..."}


USER: What is the BMI for 65 kg and 1.70 m?

ASSISTANT: {"type": "to

'Your BMI is 22.49 kg/m^2.'

# 1.4. Using libraries (LangChain + LangGraph)

In [11]:
"""## 1.4. Using libraries (LangChain + LangGraph)"""
!pip install langgraph langchain langchain-openai -q
from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.8 MB/s eta 0:00:00


In [12]:

@tool
def langchain_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI)."""
    return calculate_bmi(weight_kg, height_m)

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

llm = ChatOpenAI(model=MODEL, temperature=0).bind_tools([langchain_bmi])

def llm_node(state: AgentState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

tools_node = ToolNode([langchain_bmi])

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tools_node)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")
app = graph.compile()

# 🩺 Medical test with LangGraph
result = app.invoke({
    "messages": [
        SystemMessage(content="You are a medical assistant. Always try to use provided tools."),
        HumanMessage(content="What is the BMI for a patient weighing 90 kg and 1.85 m tall?")
    ]
})
print(result["messages"][-1].content)

# Streaming demo
for step in app.stream(
    {
        "messages": [
            SystemMessage(content="You are a medical assistant. Always try to use provided tools."),
            HumanMessage(content="Calculate BMI for 75 kg and 1.78 m.")
        ]
    },
    stream_mode="values",
):
    last = step["messages"][-1]
    print("NODE OUTPUT:")
    print(type(last).__name__)
    if hasattr(last, "content"):
        print("Content:", last.content)
    if hasattr(last, "tool_calls") and last.tool_calls:
        print("Tool calls:", last.tool_calls)
    print("==========")

BMI = 26.3 kg/m^2

Category: Overweight (25.0–29.9)

Note: BMI is a screening measure, not a diagnosis. For personalized guidance or concerns, consult a healthcare provider.
NODE OUTPUT:
HumanMessage
Content: Calculate BMI for 75 kg and 1.78 m.
NODE OUTPUT:
AIMessage
Content: 
Tool calls: [{'name': 'langchain_bmi', 'args': {'weight_kg': 75, 'height_m': 1.78}, 'id': 'call_OU3ui9YJ9AwuS1YZSnGnsWEo', 'type': 'tool_call'}]
NODE OUTPUT:
ToolMessage
Content: 23.67
NODE OUTPUT:
AIMessage
Content: BMI = 23.67 kg/m^2 (about 23.7)

Interpretation: Normal weight range (18.5–24.9).

Note: BMI is a rough estimate and doesn’t distinguish between muscle and fat. Consider other health factors for a complete assessment.
